# 06 · Multi-Agent Systems: Supervisors and Handoffs

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama. This notebook makes many
> model calls, so expect a few minutes on a local model.

One agent with twenty tools and a long prompt gets confused. Several **focused
agents**, each with a few tools and a clear job, coordinated by a graph, usually
do better. This notebook builds the two main architectures from scratch:
a **supervisor** that delegates, and **handoffs** where agents pass control to
each other.

**You will learn**
- When multiple agents help, and when they only add cost
- **Supervisor (router)**: a coordinator picks which worker runs next, in a loop
- **Supervisor (agents as tools)**: workers are wrapped as tools for a coordinating agent
- **Handoffs**: an agent transfers control with `Command(graph=Command.PARENT)`
- **Context isolation**: what each agent should see, and why it matters
- How to choose between the architectures

**Prerequisites:** notebooks 04 (agents) and 05 (`Command`).

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import time
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from langgraph.prebuilt import InjectedState
from langchain.agents import create_agent
from common import get_model

model = get_model()


def show(graph, question, **kwargs):
    # Stream a run and print who said what.
    for update in graph.stream({'messages': [('user', question)]}, stream_mode='updates', **kwargs):
        for node, delta in update.items():
            for m in (delta or {}).get('messages', []):
                calls = ' '.join(f"→{c['name']}({c['args']})" for c in getattr(m, 'tool_calls', []))
                print(f'[{node:<10}] {m.type:<5} {(m.content or "")[:120]} {calls}'.rstrip())

## 1 · Why (and when) multiple agents

A single agent's prompt and tool list grow with every capability you add.
Past a point it picks the wrong tool, forgets instructions, and every call pays
for a huge context. Splitting work across agents helps because of:

- **Specialisation.** Each agent gets a short prompt and only the tools it needs.
- **Context isolation.** A worker sees its task, not the whole conversation, so
  it's cheaper and less distracted.
- **Separate ownership.** Different people (or teams) can build and test different agents.

But multi-agent systems **cost more calls** (coordination isn't free), are
harder to debug, and can bounce work between agents. The rule: **start with one
agent; split when a single agent demonstrably struggles.**

## 2 · The architectures

```
  SUPERVISOR (router)             SUPERVISOR (agents as tools)          HANDOFFS
  ───────────────────             ────────────────────────────          ────────
       supervisor                       coordinator agent                triage ──► billing
      ▲   │   ▲                       tools: [ask_math,                    │
      │   ▼   │                               ask_travel]                  └────► tech
   math    travel                      (workers run inside tool calls)   (control moves; the
  (loop until FINISH)                                                     active agent talks)
```

| | Supervisor (router) | Agents as tools | Handoffs |
|---|---|---|---|
| Who talks to the user at the end | the supervisor | the coordinator | whichever agent has control |
| Control returns to the centre | after every worker | after every tool call | no, it moves |
| Built with | a routing node + worker nodes | `create_agent` with wrapper tools | `Command(goto=..., graph=PARENT)` |
| Good for | multi-step tasks needing several specialists | "call an expert" subtasks | triage, customer-service flows |

We'll use two small specialists throughout: a **math** agent and a **travel** agent.

In [ ]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@tool
def get_weather(city: str) -> str:
    """Current weather for a city."""
    return {'pune': '31°C humid', 'oslo': '4°C rain', 'lima': '19°C cloudy'}.get(city.lower(), '20°C clear')

@tool
def flight_price(origin: str, destination: str) -> str:
    """Economy flight price in EUR for ONE passenger between two cities."""
    return f'{origin}→{destination}: 420 EUR per person'


math_agent = create_agent(model, [multiply, add], name='math',
                          system_prompt='You are a math specialist. Always use your tools; reply with the result.')
travel_agent = create_agent(model, [get_weather, flight_price], name='travel',
                            system_prompt='You are a travel specialist. Use your tools; reply briefly.')

## 3 · Supervisor (router)

A supervisor node reads the conversation and decides, via structured output,
which worker should act next, or `FINISH`. Each worker runs, reports back, and
control returns to the supervisor. It's the orchestrator-workers pattern with a
loop.

Two design decisions matter:

- **Context isolation.** A worker node doesn't hand the whole conversation to
  its agent. It sends a focused **task** written by the supervisor, and adds only
  the worker's **final answer** back to the shared conversation. Workers stay
  cheap; the shared history stays readable.
- **A guard.** The supervisor has a turn budget, like every loop in notebook 05.
- **Robust decisions.** Structured output can come back empty (the model answers
  in prose instead of filling the schema). While building this notebook, that
  crashed the supervisor on its third turn. `include_raw=True` returns `None`
  instead of raising, so the node retries once and otherwise finishes safely.

In [ ]:
class Decision(BaseModel):
    next: Literal['math', 'travel', 'FINISH']
    task: str                                   # instructions for the chosen worker


class TeamState(MessagesState):
    next: str
    task: str
    turns: int


SUPERVISOR_PROMPT = """You coordinate two specialists:
- math: arithmetic and calculations
- travel: weather and flight prices
Look at the conversation. Handle ONE sub-question per turn: pick the specialist for the
next unanswered part and write them a precise, self-contained task (include any numbers
they need). Never do arithmetic yourself: send it to math. When every part of the user's
request has been answered by a specialist, choose FINISH."""

decider = model.with_structured_output(Decision, method='function_calling', include_raw=True)
MAX_TURNS = 5


def supervisor(state: TeamState) -> dict:
    if state.get('turns', 0) >= MAX_TURNS:                        # the guard
        return {'next': 'FINISH'}
    prompt = [('system', SUPERVISOR_PROMPT)] + state['messages']
    d = decider.invoke(prompt)['parsed']
    if d is None:                                   # the model produced no valid decision:
        d = decider.invoke(prompt + [('user', 'Reply by calling Decision.')])['parsed']   # retry once
    if d is None:                                   # still nothing: stop safely instead of crashing
        return {'next': 'FINISH'}
    print(f"  supervisor → {d.next}: {d.task[:80]}")
    return {'next': d.next, 'task': d.task, 'turns': state.get('turns', 0) + 1}


def worker_node(agent, name):
    def run(state: TeamState) -> dict:
        result = agent.invoke({'messages': [('user', state['task'])]})    # isolated context
        answer = result['messages'][-1].content
        return {'messages': [AIMessage(f'[{name}] {answer}', name=name)]}  # only the final answer
    return run


def final_answer(state: TeamState) -> dict:
    reply = model.invoke([('system', 'Combine the specialists\' results into the final answer. Be brief.')]
                         + state['messages']
                         + [('user', 'Now write the final answer to my original question.')])   # end on a user turn
    return {'messages': [reply]}


b = StateGraph(TeamState)
b.add_node('supervisor', supervisor)
b.add_node('math', worker_node(math_agent, 'math'))
b.add_node('travel', worker_node(travel_agent, 'travel'))
b.add_node('final_answer', final_answer)
b.add_edge(START, 'supervisor')
b.add_conditional_edges('supervisor', lambda s: 'final_answer' if s['next'] == 'FINISH' else s['next'],
                        ['math', 'travel', 'final_answer'])
b.add_edge('math', 'supervisor')
b.add_edge('travel', 'supervisor')
b.add_edge('final_answer', END)
team = b.compile()

print(team.get_graph().draw_ascii())

In [ ]:
t0 = time.perf_counter()
out = team.invoke({'messages': [('user', 'I fly Pune to Oslo for 3 people. What is the total ticket cost, '
                                         'and what weather should we pack for?')]})
for m in out['messages']:
    print(f'{m.type:<5} {m.content[:200]}')
print(f'\n{time.perf_counter() - t0:.0f}s')

*(Why does `final_answer` end with an extra user message? The history ends with
the specialists' AI messages, and many models return an **empty reply** when
the last message is their own. Ending on a user turn is a reliable fix.)*

Follow the trace. The supervisor asked **travel** for the price, then gave
**math** a task containing the number it had just learned (price × 3), asked
travel about the weather, then chose FINISH. (Runs vary, but that's the shape.)

**Two prompt details mattered.** An earlier version said "flight price" without
"per person", so the travel agent reported 420 EUR as the total for three
people and the supervisor happily finished, never calling math. It also let the
supervisor answer several parts at once, and the final answer then *invented*
the weather. Precise tool descriptions ("for ONE passenger") and a
one-sub-question-per-turn rule fixed both. The workers never saw each other's conversations,
only the tasks the supervisor wrote. That's context isolation, and it's why
the supervisor's task-writing quality matters so much.

## 4 · Supervisor (agents as tools)

A lighter variant: wrap each worker agent in a **tool**, and give those tools
to an ordinary `create_agent` coordinator. Tool calling *is* the delegation:

In [ ]:
@tool
def ask_math_expert(task: str) -> str:
    """Delegate a calculation to the math specialist. Describe the full calculation."""
    return math_agent.invoke({'messages': [('user', task)]})['messages'][-1].content

@tool
def ask_travel_expert(task: str) -> str:
    """Delegate a weather or flight-price question to the travel specialist."""
    return travel_agent.invoke({'messages': [('user', task)]})['messages'][-1].content


coordinator = create_agent(model, [ask_math_expert, ask_travel_expert], name='coordinator',
                           system_prompt='You coordinate specialists. Delegate every fact or calculation; '
                                         'never compute or guess yourself. Then answer briefly.')
show(coordinator, 'Flights Lima to Oslo for 2 people: total cost? And the weather in Lima?')

This is often the **simplest multi-agent design that works**: no custom graph,
just tools. The trade-off is visibility. The workers' internal steps happen
inside a tool call, so they don't appear as separate graph steps (they do appear
with `subgraphs=True` streaming and in traces, notebook 09).

## 5 · Handoffs: agents pass control

In a handoff architecture there's no central coordinator. The **active agent**
decides to transfer the conversation to another agent, which then talks to the
user directly. Customer-service triage is the classic example.

A handoff is a **tool** that returns a `Command`:

```python
Command(goto='billing', graph=Command.PARENT, update={'messages': [...]})
```

- `goto='billing'`: the next node to run.
- `graph=Command.PARENT`: the tool runs *inside* the triage agent (a subgraph),
  so the jump has to happen in the **parent** graph, where the `billing` node lives.
- `update=`: messages to add to the parent's state. We include **both** the AI
  message that requested the transfer **and** a `ToolMessage` answering it. A
  tool result without its request is malformed history, and some providers
  reject it.

In [ ]:
def make_handoff(agent_name: str, description: str):
    @tool(f'transfer_to_{agent_name}', description=description)
    def handoff(state: Annotated[dict, InjectedState],                 # the calling agent's state
                tool_call_id: Annotated[str, InjectedToolCallId]) -> Command:
        request = state['messages'][-1]                                # the AI message with this tool call
        done = ToolMessage(f'Transferred to {agent_name}.', tool_call_id=tool_call_id)
        return Command(goto=agent_name, graph=Command.PARENT, update={'messages': [request, done]})
    return handoff


@tool
def issue_refund(order_id: str) -> str:
    """Refund an order."""
    return f'Refund issued for order {order_id}.'

@tool
def reset_password(email: str) -> str:
    """Send a password-reset link."""
    return f'Reset link sent to {email}.'


def with_clean_context(agent):
    # Run an agent on the conversation MINUS other agents' tool traffic.
    def run(state: MessagesState) -> dict:
        visible = [m for m in state['messages']
                   if m.type == 'human' or (m.type == 'ai' and not m.tool_calls)]
        result = agent.invoke({'messages': visible})
        return {'messages': result['messages'][len(visible):]}    # only what THIS agent added
    return run


triage = create_agent(model, name='triage',
    tools=[make_handoff('billing', 'Transfer refund, charge and invoice issues to billing.'),
           make_handoff('tech', 'Transfer login and technical issues to tech support.')],
    system_prompt='You are the triage desk. Never solve problems yourself: transfer to the right specialist.')
billing = create_agent(model, [issue_refund], name='billing',
    system_prompt='You are billing support. Resolve the customer issue with your tools, then reply to them.')
tech = create_agent(model, [reset_password], name='tech',
    system_prompt='You are tech support. Resolve the issue with your tools, then reply to the customer.')

b = StateGraph(MessagesState)
b.add_node('triage', triage, destinations=('billing', 'tech'))   # destinations: for drawing only
b.add_node('billing', with_clean_context(billing))
b.add_node('tech', with_clean_context(tech))
b.add_edge(START, 'triage')
helpdesk = b.compile()

print(helpdesk.get_graph().draw_ascii())
show(helpdesk, 'Please refund order A-17, it arrived broken.')
print()
show(helpdesk, 'I cannot log in, my email is sam@example.com.')

**Why `with_clean_context`?** The first version added the specialists as
plain nodes, so they saw the whole shared history, including triage's
`transfer_to_billing` call and its result. The billing agent **imitated** it and
called `transfer_to_billing` itself, a tool it doesn't even have, and tech
support never reset the password. A handoff is also a **context decision**. The
shared history stays complete and well-formed, but each receiving agent sees a
filtered view: the customer's words and final answers, not other agents' tool
traffic.

Notice there's no edge from `billing` or `tech` anywhere: once an agent has
control, **it** finishes the conversation. That's what makes handoffs feel
natural to users ("let me transfer you"), and also what makes them harder to
control. No coordinator checks the work afterwards.

`destinations=('billing', 'tech')` doesn't affect execution. It only lets
`draw_ascii` show the possible jumps, since `Command` routing is invisible to
the static graph otherwise.

## 6 · Choosing an architecture

| Question | Points to |
|---|---|
| Does the answer need **several** specialists combined? | supervisor (router or agents-as-tools) |
| Do you need a checkpoint **between** specialists (review, approval)? | supervisor (router), since each worker is a graph node |
| Is it "figure out who should handle this, then let them"? | handoffs |
| Do you just need an expert for a sub-question? | agents as tools |
| Are you unsure? | **one agent**, then split when it struggles |

**Libraries.** LangChain also publishes prebuilt helpers
(`langgraph-supervisor`, `langgraph-swarm`) that package these patterns. They
build the same graphs you built here, so once you understand this notebook you
can read, debug and customise them.

## 7 · Common mistakes

- **Splitting too early.** Three agents where one would do triples the cost and the failure modes.
- **Sharing the full history with every worker.** It costs more and distracts them. Send a focused task.
- **Vague delegation.** "Handle the math" loses the numbers. The coordinator must write complete, self-contained tasks.
- **No turn budget.** Supervisors can ping-pong between workers. Always cap turns.
- **Orphaned tool messages in handoffs.** Carry the requesting AI message along with the `ToolMessage`.
- **Receiving agents that see other agents' tool calls.** They imitate them. Filter the view at the handoff.
- **Ambiguous tool outputs** ("420 EUR": each or total?). Coordinators build on them, so say exactly what a number means.
- **Handoff loops.** Agent A transfers to B, which transfers back to A… Limit handoffs or give agents clear, non-overlapping scopes.

## 8 · Check yourself

<details><summary><b>1.</b> Give two reasons to split one agent into several, and one reason not to.</summary>

For: specialisation (short prompts and few tools per agent) and context isolation (workers see only their task, so they're cheaper and more focused). Against: every coordination step costs model calls, and the system is harder to debug. Split only when a single agent demonstrably struggles.
</details>

<details><summary><b>2.</b> In the router supervisor, what does a worker receive, and what does it return to the shared state?</summary>

It receives a focused task written by the supervisor (not the whole conversation), and it returns only its final answer as one message. That keeps workers isolated and the shared history compact.
</details>

<details><summary><b>3.</b> Why does a handoff tool return <code>Command(..., graph=Command.PARENT)</code>?</summary>

The tool executes inside the calling agent, which is a subgraph. The destination agent is a node in the parent graph, so the jump must be applied to the parent, not to the subgraph where the tool is running.
</details>

<details><summary><b>4.</b> Why include the requesting AI message in the handoff's update?</summary>

Otherwise the parent state gets a `ToolMessage` with no matching tool call, which is malformed conversation history. Some providers reject it, and it confuses the next agent.
</details>

<details><summary><b>5.</b> Supervisor or handoffs for: "answer a question that needs a price lookup and a calculation"?</summary>

A supervisor (or agents-as-tools). The answer combines two specialists' results, and a coordinator must gather and merge them. Handoffs suit "route the conversation to one owner".
</details>

## Takeaway

Multiple agents are a tool for **specialisation and context isolation**, not a
default. A **supervisor** keeps control central (as a router node or through
agents-as-tools); **handoffs** move control with `Command(graph=PARENT)`. Give
workers focused tasks, return compact results, cap turns, and keep message
history well-formed.

Next → `07_human_in_the_loop.ipynb`: pausing a graph for a human, approving
or editing tool calls, and resuming.